##Configuration

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import *

SOURCE_PATH = "/Volumes/workspace/bronze/banking_source"

CUSTOMER_PATH = f"{SOURCE_PATH}/customers.csv"
ACCOUNT_PATH = f"{SOURCE_PATH}/accounts.csv"
MERCHANT_PATH = f"{SOURCE_PATH}/merchants.csv"
DEVICE_PATH = f"{SOURCE_PATH}/devices.csv"

print("Source path:", SOURCE_PATH)

Source path: /Volumes/workspace/bronze/banking_source


##Customer Schema

In [0]:
customer_schema = StructType([
    StructField("customer_id", StringType(), False),
    StructField("first_name", StringType(), True),
    StructField("last_name", StringType(), True),
    StructField("date_of_birth", DateType(), True),
    StructField("gender", StringType(), True),
    StructField("city", StringType(), True),
    StructField("state", StringType(), True),
    StructField("country", StringType(), True),
    StructField("customer_segment", StringType(), True),
    StructField("kyc_status", StringType(), True),
    StructField("customer_since", DateType(), True),
    StructField("risk_category", StringType(), True)
])

##Reading Customers

In [0]:
customers_raw = (
    spark.read
    .format("csv")
    .option("header", "true")
    .schema(customer_schema)
    .load(CUSTOMER_PATH)
)

print("Customer records:", f"{customers_raw.count():,}")

display(customers_raw.limit(10))

Customer records: 100,000


customer_id,first_name,last_name,date_of_birth,gender,city,state,country,customer_segment,kyc_status,customer_since,risk_category
CUST000001,Sneha,Kumar,1998-04-24,F,Hyderabad,Telangana,India,REGULAR,VERIFIED,2022-06-13,LOW
CUST000002,Priya,Raju,1976-04-19,F,Bengaluru,Karnataka,India,REGULAR,VERIFIED,2017-03-29,LOW
CUST000003,Rahul,Mehta,1989-10-11,M,Bengaluru,Karnataka,India,REGULAR,VERIFIED,2024-11-06,MEDIUM
CUST000004,Vikram,Nair,1975-05-10,F,Vijayawada,Andhra Pradesh,India,REGULAR,VERIFIED,2021-07-11,LOW
CUST000005,Ananya,Raju,1982-03-23,M,Warangal,Telangana,India,PREMIUM,VERIFIED,2015-01-15,LOW
CUST000006,Aarav,Reddy,1986-10-12,F,Kolkata,West Bengal,India,VIP,VERIFIED,2021-01-25,LOW
CUST000007,Priya,Gupta,1965-04-08,M,Kochi,Kerala,India,REGULAR,VERIFIED,2022-06-20,LOW
CUST000008,Sneha,Sharma,1992-04-22,F,Visakhapatnam,Andhra Pradesh,India,VIP,VERIFIED,2021-05-02,LOW
CUST000009,Vikram,Nair,1991-05-21,F,New Delhi,Delhi,India,REGULAR,REJECTED,2024-10-08,MEDIUM
CUST000010,Sanjay,Patel,1980-06-29,F,New Delhi,Delhi,India,VIP,VERIFIED,2019-01-31,MEDIUM


##Customer Data Quality

In [0]:
customer_total = customers_raw.count()

customer_unique = (
    customers_raw
    .select("customer_id")
    .distinct()
    .count()
)

customer_null_ids = (
    customers_raw
    .filter(F.col("customer_id").isNull())
    .count()
)

print("Total records :", f"{customer_total:,}")
print("Unique IDs    :", f"{customer_unique:,}")
print("Null IDs      :", f"{customer_null_ids:,}")

assert customer_total == 100_000
assert customer_total == customer_unique
assert customer_null_ids == 0

Total records : 100,000
Unique IDs    : 100,000
Null IDs      : 0


##Creating Bronze Customers

In [0]:
(
    customers_raw
    .withColumn("_ingest_ts", F.current_timestamp())
    .withColumn("_source_file", F.col("_metadata.file_path"))
    .withColumn("_source_system", F.lit("CUSTOMER_MASTER"))
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("workspace.bronze.customers")
)

In [0]:
display(spark.table("workspace.bronze.customers").limit(10))

customer_id,first_name,last_name,date_of_birth,gender,city,state,country,customer_segment,kyc_status,customer_since,risk_category,_ingest_ts,_source_file,_source_system
CUST000001,Sneha,Kumar,1998-04-24,F,Hyderabad,Telangana,India,REGULAR,VERIFIED,2022-06-13,LOW,2026-08-22T06:47:04.110Z,dbfs:/Volumes/workspace/bronze/banking_source/customers.csv,CUSTOMER_MASTER
CUST000002,Priya,Raju,1976-04-19,F,Bengaluru,Karnataka,India,REGULAR,VERIFIED,2017-03-29,LOW,2026-08-22T06:47:04.110Z,dbfs:/Volumes/workspace/bronze/banking_source/customers.csv,CUSTOMER_MASTER
CUST000003,Rahul,Mehta,1989-10-11,M,Bengaluru,Karnataka,India,REGULAR,VERIFIED,2024-11-06,MEDIUM,2026-08-22T06:47:04.110Z,dbfs:/Volumes/workspace/bronze/banking_source/customers.csv,CUSTOMER_MASTER
CUST000004,Vikram,Nair,1975-05-10,F,Vijayawada,Andhra Pradesh,India,REGULAR,VERIFIED,2021-07-11,LOW,2026-08-22T06:47:04.110Z,dbfs:/Volumes/workspace/bronze/banking_source/customers.csv,CUSTOMER_MASTER
CUST000005,Ananya,Raju,1982-03-23,M,Warangal,Telangana,India,PREMIUM,VERIFIED,2015-01-15,LOW,2026-08-22T06:47:04.110Z,dbfs:/Volumes/workspace/bronze/banking_source/customers.csv,CUSTOMER_MASTER
CUST000006,Aarav,Reddy,1986-10-12,F,Kolkata,West Bengal,India,VIP,VERIFIED,2021-01-25,LOW,2026-08-22T06:47:04.110Z,dbfs:/Volumes/workspace/bronze/banking_source/customers.csv,CUSTOMER_MASTER
CUST000007,Priya,Gupta,1965-04-08,M,Kochi,Kerala,India,REGULAR,VERIFIED,2022-06-20,LOW,2026-08-22T06:47:04.110Z,dbfs:/Volumes/workspace/bronze/banking_source/customers.csv,CUSTOMER_MASTER
CUST000008,Sneha,Sharma,1992-04-22,F,Visakhapatnam,Andhra Pradesh,India,VIP,VERIFIED,2021-05-02,LOW,2026-08-22T06:47:04.110Z,dbfs:/Volumes/workspace/bronze/banking_source/customers.csv,CUSTOMER_MASTER
CUST000009,Vikram,Nair,1991-05-21,F,New Delhi,Delhi,India,REGULAR,REJECTED,2024-10-08,MEDIUM,2026-08-22T06:47:04.110Z,dbfs:/Volumes/workspace/bronze/banking_source/customers.csv,CUSTOMER_MASTER
CUST000010,Sanjay,Patel,1980-06-29,F,New Delhi,Delhi,India,VIP,VERIFIED,2019-01-31,MEDIUM,2026-08-22T06:47:04.110Z,dbfs:/Volumes/workspace/bronze/banking_source/customers.csv,CUSTOMER_MASTER


## Accounts Schema

In [0]:
account_schema = StructType([
    StructField("account_id", StringType(), False),
    StructField("customer_id", StringType(), False),
    StructField("account_type", StringType(), True),
    StructField("status", StringType(), True),
    StructField("currency", StringType(), True),
    StructField("branch_code", StringType(), True),
    StructField("opening_date", DateType(), True)
])

In [0]:
accounts_raw = (
    spark.read
    .format("csv")
    .option("header", "true")
    .schema(account_schema)
    .load(ACCOUNT_PATH)
)

print("Account records:", f"{accounts_raw.count():,}")

display(accounts_raw.limit(10))

Account records: 150,000


account_id,customer_id,account_type,status,currency,branch_code,opening_date
ACC000001,CUST014003,SAVINGS,ACTIVE,INR,CHE001,2023-07-28
ACC000002,CUST040067,SAVINGS,DORMANT,INR,HYD001,2021-09-03
ACC000003,CUST024143,SAVINGS,ACTIVE,INR,KOL001,2021-11-16
ACC000004,CUST026129,CURRENT,ACTIVE,INR,MUM001,2019-05-13
ACC000005,CUST033626,SAVINGS,CLOSED,INR,HYD001,2016-09-20
ACC000006,CUST040901,SAVINGS,ACTIVE,INR,CHE001,2020-03-25
ACC000007,CUST055285,SALARY,ACTIVE,INR,BLR001,2025-01-10
ACC000008,CUST038473,SALARY,ACTIVE,INR,VIZ001,2016-01-06
ACC000009,CUST017541,SAVINGS,ACTIVE,INR,VIZ001,2021-10-08
ACC000010,CUST008838,SAVINGS,ACTIVE,INR,CHE001,2024-10-02


In [0]:
invalid_accounts = (
    accounts_raw
    .select("account_id", "customer_id")
    .join(
        customers_raw.select("customer_id"),
        "customer_id",
        "left_anti"
    )
)

invalid_account_count = invalid_accounts.count()

print("Accounts with invalid customer_id:", invalid_account_count)

assert invalid_account_count == 0

Accounts with invalid customer_id: 0


##Writing Accounts to Bronze

In [0]:
(
    accounts_raw
    .withColumn("_ingest_ts", F.current_timestamp())
    .withColumn("_source_file", F.col("_metadata.file_path"))
    .withColumn("_source_system", F.lit("ACCOUNT_MASTER"))
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("workspace.bronze.accounts")
)

##Merchants

In [0]:
merchant_schema = StructType([
    StructField("merchant_id", StringType(), False),
    StructField("merchant_name", StringType(), True),
    StructField("merchant_category", StringType(), True),
    StructField("city", StringType(), True),
    StructField("state", StringType(), True),
    StructField("country", StringType(), True),
    StructField("status", StringType(), True)
])

In [0]:
merchants_raw = (
    spark.read
    .format("csv")
    .option("header", "true")
    .schema(merchant_schema)
    .load(MERCHANT_PATH)
)

print("Merchant records:", f"{merchants_raw.count():,}")

display(merchants_raw.limit(10))

Merchant records: 25,000


merchant_id,merchant_name,merchant_category,city,state,country,status
MER00001,Merchant_00001,RETAIL,Jaipur,Rajasthan,India,ACTIVE
MER00002,Merchant_00002,FOOD,Bengaluru,Karnataka,India,INACTIVE
MER00003,Merchant_00003,GROCERY,Kolkata,West Bengal,India,ACTIVE
MER00004,Merchant_00004,FUEL,Kochi,Kerala,India,ACTIVE
MER00005,Merchant_00005,FUEL,Bengaluru,Karnataka,India,ACTIVE
MER00006,Merchant_00006,GROCERY,Jaipur,Rajasthan,India,ACTIVE
MER00007,Merchant_00007,RETAIL,Pune,Maharashtra,India,ACTIVE
MER00008,Merchant_00008,FOOD,Kolkata,West Bengal,India,ACTIVE
MER00009,Merchant_00009,RETAIL,Kochi,Kerala,India,ACTIVE
MER00010,Merchant_00010,ENTERTAINMENT,Visakhapatnam,Andhra Pradesh,India,ACTIVE


In [0]:
(
    merchants_raw
    .withColumn("_ingest_ts", F.current_timestamp())
    .withColumn("_source_file", F.col("_metadata.file_path"))
    .withColumn("_source_system", F.lit("MERCHANT_MASTER"))
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("workspace.bronze.merchants")
)

##Devices

In [0]:
device_schema = StructType([
    StructField("device_id", StringType(), False),
    StructField("customer_id", StringType(), False),
    StructField("device_type", StringType(), True),
    StructField("os", StringType(), True),
    StructField("status", StringType(), True),
    StructField("registered_date", DateType(), True)
])

In [0]:
devices_raw = (
    spark.read
    .format("csv")
    .option("header", "true")
    .schema(device_schema)
    .load(DEVICE_PATH)
)

print("Device records:", f"{devices_raw.count():,}")

display(devices_raw.limit(10))

Device records: 75,000


device_id,customer_id,device_type,os,status,registered_date
DEV000001,CUST060987,ATM,ANDROID,ACTIVE,2026-03-12
DEV000002,CUST074107,WEB,WINDOWS,ACTIVE,2022-09-26
DEV000003,CUST069653,ANDROID,WINDOWS,ACTIVE,2018-06-18
DEV000004,CUST036678,IOS,WINDOWS,ACTIVE,2020-02-08
DEV000005,CUST018639,ATM,LINUX,BLOCKED,2025-03-30
DEV000006,CUST011542,ANDROID,OTHER,ACTIVE,2018-09-15
DEV000007,CUST072568,ATM,LINUX,ACTIVE,2024-06-27
DEV000008,CUST012028,IOS,IOS,ACTIVE,2020-04-14
DEV000009,CUST002809,WEB,ANDROID,ACTIVE,2025-10-16
DEV000010,CUST063722,IOS,OTHER,ACTIVE,2018-04-22


In [0]:
invalid_devices = (
    devices_raw
    .select("device_id", "customer_id")
    .join(
        customers_raw.select("customer_id"),
        "customer_id",
        "left_anti"
    )
)

invalid_device_count = invalid_devices.count()

print("Devices with invalid customer_id:", invalid_device_count)

assert invalid_device_count == 0

Devices with invalid customer_id: 0


In [0]:
(
    devices_raw
    .withColumn("_ingest_ts", F.current_timestamp())
    .withColumn("_source_file", F.col("_metadata.file_path"))
    .withColumn("_source_system", F.lit("DEVICE_MASTER"))
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("workspace.bronze.devices")
)


In [0]:
tables = [
    "workspace.bronze.customers",
    "workspace.bronze.accounts",
    "workspace.bronze.merchants",
    "workspace.bronze.devices"
]

for table in tables:
    count = spark.table(table).count()
    print(f"{table:<40} {count:>10,} records")

workspace.bronze.customers                  100,000 records
workspace.bronze.accounts                   150,000 records
workspace.bronze.merchants                   25,000 records
workspace.bronze.devices                     75,000 records
